<a href="https://colab.research.google.com/github/eduzegarra/grade_01/blob/main/ENA_integra.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import os
import glob

In [ ]:
os.chdir('/content/drive/MyDrive/a_SENAMHI/ENA/')
!ls

### Loading .dta files into a dictionary

I will now iterate through the previously defined list of directories, navigate into each, and load any `.dta` files found into a dictionary named `dta_dataframes`. Each DataFrame will be stored with its filename (without the `.dta` extension) as the key.

In [ ]:
# Initialize an empty dictionary to store the dataframes, nested by year
dta_dataframes = {}

# Save the original working directory
original_cwd = os.getcwd()

print("Loading .dta files...\n")

years = range(2023, 2026) # Define the range of years

base_path = '/content/drive/MyDrive/a_SENAMHI/ENA/'

for year in years:
    year_path = os.path.join(base_path, str(year))
    if not os.path.isdir(year_path):
        print(f"Warning: Year directory not found - {year_path}")
        continue

    print(f"Processing year: {year}")
    dta_dataframes[year] = {} # Initialize dictionary for the current year

    # Get subdirectories within the year directory that contain 'Modulo' in their name
    module_dirs = [d for d in os.listdir(year_path) if os.path.isdir(os.path.join(year_path, d))
              and 'Modulo' in d]

    if not module_dirs:
        print(f"No 'Modulo' subdirectories found in {year_path}")
        continue

    for module_dir in module_dirs:
        full_module_path = os.path.join(year_path, module_dir)

        # Change to the module directory temporarily
        os.chdir(full_module_path)

        # Find all .dta files in the current directory
        stata_files = glob.glob('*.dta')

        if not stata_files:
            # print(f"No .dta files found in {full_module_path}") # This can be noisy
            continue

        for file_name in stata_files:
            try:
                # Construct the full file path (relative to the current working directory)
                file_path = os.path.join(os.getcwd(), file_name)
                df = pd.read_stata(file_path)

                # Use the filename without extension as the dictionary key, nested under the year
                key_name = os.path.splitext(file_name)[0]
                dta_dataframes[year][key_name] = df
                print(f"Loaded {file_name} as '{key_name}' from {year}/{module_dir}")
            except Exception as e:
                print(f"Error loading {file_name} from {year}/{module_dir}: {e}")

# Restore the original working directory
os.chdir(original_cwd)

total_dataframes = sum(len(dfs) for dfs in dta_dataframes.values())
print(f"\nSuccessfully loaded {total_dataframes} DataFrames in total.")
print("\nAvailable DataFrames (keys in 'dta_dataframes' dictionary):")
for year, dfs in dta_dataframes.items():
    print(f"Year {year}:")
    for key in dfs.keys():
        print(f"  - {key}")

# Example: Display the head of the first loaded DataFrame (if any)
# if dta_dataframes:
#    first_year = list(dta_dataframes.keys())[0]
#    if dta_dataframes[first_year]:
#        first_key = list(dta_dataframes[first_year].keys())[0]
#        print(f"\nDisplaying the first 5 rows of '{first_key}' from year {first_year}:")
#        display(dta_dataframes[first_year][first_key].head())


In [ ]:
print('Keys in dta_dataframes:')
for year, dataframes in dta_dataframes.items():
    print(f'  Year {year}: {list(dataframes.keys())}')

In [ ]:
caratula_23=dta_dataframes[2023]['01_CARATULA']
caratula_24=dta_dataframes[2024]['CARATULA']
caratula_25=dta_dataframes[2025]['01_CARATULA']

In [ ]:
caratulas = pd.concat([caratula_23, caratula_24, caratula_25], ignore_index=True)

In [ ]:
info.filter(['ANIO','NSEGM','ID_PROD','UA']).info()

In [ ]:
caratulas['cod_prod']=info['ANIO'].astype(str) + '_'+info.NSEGM+info.ID_PROD+info.UA

In [ ]:
caratulas.cod_prod.isna().sum()

In [ ]:
caratulas.to_stata('/content/drive/MyDrive/a_SENAMHI/OUT/caratulas.dta')

In [ ]:
caratulas.info()

# Modulo de informacion

In [ ]:
info_23=dta_dataframes[2023]['15_CAP700']
info_24=dta_dataframes[2024]['15_CAP700']
info_25=dta_dataframes[2025]['15_CAP700']

In [ ]:
info=pd.concat([info_23,info_24,info_25], ignore_index=True)

In [ ]:
ll=['1','2','3','4','5','6_1','6_2','6_3']

In [ ]:
new_cols_to_add = {}
for vv in ll:
    col_name_source = 'P707_' + vv
    col_name_target = 'info_' + vv

    if col_name_source in info.columns:
        new_cols_to_add[col_name_target] = info[col_name_source].map({'Sí': 1, 'No': 0}).astype(pd.Int8Dtype())
    else:
        print(f"Warning: Column {col_name_source} not found in info DataFrame. Initializing '{col_name_target}' with NA values.")
        new_cols_to_add[col_name_target] = pd.Series(pd.NA, index=info.index, dtype=pd.Int8Dtype())

# Create a DataFrame from the dictionary of new columns
new_columns_df = pd.DataFrame(new_cols_to_add, index=info.index)

# Concatenate the new columns to the original info DataFrame
info = pd.concat([info, new_columns_df], axis=1)

In [ ]:
info_cols = ['info_' + str(i) for i in ll]
info[info_cols].sum()

# PROCEDENCIA DE LA INFORMACIÓN

In [ ]:
## MINISTERIO DE AGRICULTURA
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['inst_MIDAGRI_0'+kk]=info['P708_'+kk+'_1'].str.contains('Ministerio', na=False).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'inst_MIDAGRI_0'+kk]=np.nan

## COMERCIANTE O AMIGO
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['inst_AMIGO_0'+kk]=info['P708_'+kk+'_7'].str.contains('Comerciante', na=False).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'inst_AMIGO_0'+kk]=np.nan

## SENAMHI
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['inst_SENAMHI_0'+kk]=info['P708_'+kk+'_9'].str.contains('Servicio', na=False).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'inst_SENAMHI_0'+kk]=np.nan

## EMPRESA
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['inst_EMPRESA_0'+kk]=info['P708_'+kk+'_5'].str.contains('Empresa', na=False).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'inst_EMPRESA_0'+kk]=np.nan

## GOBIERNO REGIONAL
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['inst_GORE_0'+kk]=info['P708_'+kk+'_2'].str.contains('Regional', na=False).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'inst_GORE_0'+kk]=np.nan

## MUNICIPIO
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['inst_MUNI_0'+kk]=info['P708_'+kk+'_3'].str.contains('Local', na=False).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'inst_MUNI_0'+kk]=np.nan

## ASOCIACIÓN DE PRODUCTORES
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['inst_ASOC_0'+kk]=info['P708_'+kk+'_6'].str.contains('Asociación', na=False).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'inst_ASOC_0'+kk]=np.nan

In [ ]:
info.filter(regex=r'^inst_').sum()
#describe(include='all').transpose()

# MEDIOS DE ACCESO A LA INFORMACIÓN

In [ ]:
## Radio
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['medio_RADIO_0'+kk]=(info['P709_'+kk+'_1']==
            info['P709_'+kk+'_1'].cat.categories[1]).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'medio_RADIO_0'+kk]=np.nan

## TV
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['medio_TV_0'+kk]=(info['P709_'+kk+'_2']==
            info['P709_'+kk+'_2'].cat.categories[1]).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'medio_TV_0'+kk]=np.nan

## INTERNET
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['medio_INTERNET_0'+kk]=(info['P709_'+kk+'_6']==
            info['P709_'+kk+'_6'].cat.categories[1]).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'medio_INTERNET_0'+kk]=np.nan

## TELEFONO
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['medio_TELEF_0'+kk]=(info['P709_'+kk+'_3']==
            info['P709_'+kk+'_3'].cat.categories[1]).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'medio_TELEF_0'+kk]=np.nan

## VERBAL
import numpy as np
ll=['1','2','3','4','5','6_1','6_2','6_3']
for kk in ll:
  info['medio_VERBAL_0'+kk]=(info['P709_'+kk+'_8']==
            info['P709_'+kk+'_8'].cat.categories[1]).astype(int)
for kk in ll:
  info.loc[(info['info_'+kk]==0), 'medio_VERBAL_0'+kk]=np.nan

In [ ]:
info.filter(regex=r'^medio_').sum()
#describe(include='all').transpose()

In [ ]:
info.filter(regex=r'^P710').info()

# NECESIDADES DE INFORMACIÓN

In [ ]:
info['neces_AGROCLIM']=info['P710_1'].str.contains('agroclimática', na=False).astype(int)
info['neces_PLAGAS']=(info['P710_2'].str.contains('plagas', na=False)).astype(int)
info['neces_PRECIO_INSUMOS']=(info['P710_3'].str.contains('insumos', na=False)).astype(int)
info['neces_PRECIO_VENTA']=(info['P710_4'].str.contains('venta', na=False)).astype(int)

In [ ]:
info.filter(regex=r'^neces_').describe(include='all').transpose()

In [ ]:
info['ANIO']=info['ANIO'].astype(int)
info['cod_prod']=info['ANIO'].astype(str) + '_'+info.NSEGM+info.ID_PROD+info.UA

In [ ]:
info=info.filter(regex=r'cod_prod|^info|^inst_|^medio_|^neces_')

In [ ]:
info.to_stata('/content/drive/MyDrive/a_SENAMHI/OUT/informacion.dta')

In [ ]:
info.shape

In [ ]:
## SUPERFICIE DE LA UNIDAD AGROPECUARIA
supag_23 = dta_dataframes[2023]['02_CAP100B_02'].copy()
print(supag_23.shape)
supag_23=supag_23[supag_23['P142_1'].str.contains('¿Agrícola?', na=False)]
print(supag_23.shape)
supag_23.ANIO=supag_23.ANIO.astype('int')
supag_23['cod_prod'] = supag_23['ANIO'].astype(str) + '_'+supag_23.NSEGM+supag_23.ID_PROD+supag_23.UA
supag_23 = supag_23.groupby('cod_prod', as_index=False)['P147_SUP_ha'].sum().rename(columns={'P147_SUP_ha':'t_uahas'})

supag_24 = dta_dataframes[2024]['02_CAP100B_02'].copy()
print(supag_24.shape)
supag_24=supag_24[supag_24['P142_1'].str.contains('¿Agrícola?', na=False)]
print(supag_24.shape)
supag_24.ANIO=supag_24.ANIO.astype('int')
supag_24['cod_prod'] = supag_24['ANIO'].astype(str) + '_'+supag_24.NSEGM+supag_24.ID_PROD+supag_24.UA
supag_24 = supag_24.groupby('cod_prod', as_index=False)['P147_SUP_ha'].sum().rename(columns={'P147_SUP_ha':'t_uahas'})

supag_25 = dta_dataframes[2025]['02_CAP100B_02'].copy()
print(supag_25.shape)
supag_25=supag_25[supag_25['P142_1'].str.contains('¿Agrícola?', na=False)]
print(supag_25.shape)
supag_25.ANIO=supag_25.ANIO.astype('int')
supag_25['cod_prod'] = supag_25['ANIO'].astype(str) + '_'+supag_25.NSEGM+supag_25.ID_PROD+supag_25.UA
supag_25 = supag_25.groupby('cod_prod', as_index=False)['P147_SUP_ha'].sum().rename(columns={'P147_SUP_ha':'t_uahas'})


In [219]:
supag=pd.concat([supag_23,supag_24,supag_25], ignore_index=True)

In [223]:
supag.to_stata('/content/drive/MyDrive/a_SENAMHI/OUT/supag.dta')

In [285]:
# MAQUINAS Y EQUIPOS (NO HAY ESTE MODULO PARA ANTES DE 2022)

mq_eq_23 = dta_dataframes[2023]['21_CAP1200B_ME'].copy()
mq_eq_23['cod_prod'] = mq_eq_23['ANIO'].astype(str) + '_'+mq_eq_23.NSEGM + mq_eq_23.ID_PROD + mq_eq_23.UA
mq_eq_23['t_usomaq'] = (mq_eq_23['P1207_TIPO'].str.contains('Maquinaria', na=False)).astype(int)
mq_eq_23['t_usoequ'] = (mq_eq_23['P1207_TIPO'].str.contains('Equipo', na=False)).astype(int)
mq_eq_23['t_maqpro'] = ((mq_eq_23['P1208'].str.contains(pat=r'^Propio', na=False)) & (mq_eq_23['t_usomaq']==1)).astype(int)
mq_eq_23['t_maqalq'] = (mq_eq_23['P1208'].str.contains(pat=r'^Alquilado', na=False) & (mq_eq_23['t_usomaq']==1)).astype(int)
mq_eq_23=mq_eq_23.groupby('cod_prod', as_index=False)[['t_usomaq','t_usoequ','t_maqpro','t_maqalq']].max()

mq_eq_24 = dta_dataframes[2024]['21_CAP1200B_ME'].copy()
mq_eq_24['cod_prod'] = mq_eq_24['ANIO'].astype(str) + '_'+mq_eq_24.NSEGM + mq_eq_24.ID_PROD + mq_eq_24.UA
mq_eq_24['t_usomaq'] = (mq_eq_24['P1207_TIPO'].str.contains('Maquinaria', na=False)).astype(int)
mq_eq_24['t_usoequ'] = (mq_eq_24['P1207_TIPO'].str.contains('Equipo', na=False)).astype(int)
mq_eq_24['t_maqpro'] = ((mq_eq_24['P1208'].str.contains(pat=r'^Propio', na=False)) & (mq_eq_24['t_usomaq']==1)).astype(int)
mq_eq_24['t_maqalq'] = (mq_eq_24['P1208'].str.contains(pat=r'^Alquilado', na=False) & (mq_eq_24['t_usomaq']==1)).astype(int)
mq_eq_24=mq_eq_24.groupby('cod_prod', as_index=False)[['t_usomaq','t_usoequ','t_maqpro','t_maqalq']].max()

mq_eq_25 = dta_dataframes[2025]['21_CAP1200B_ME'].copy()
mq_eq_25['cod_prod'] = mq_eq_25['ANIO'].astype(str) + '_'+mq_eq_25.NSEGM + mq_eq_25.ID_PROD + mq_eq_25.UA
mq_eq_25['t_usomaq'] = (mq_eq_25['P1207_TIPO'].str.contains('Maquinaria', na=False)).astype(int)
mq_eq_25['t_usoequ'] = (mq_eq_25['P1207_TIPO'].str.contains('Equipo', na=False)).astype(int)
mq_eq_25['t_maqpro'] = ((mq_eq_25['P1208'].str.contains(pat=r'^Propio', na=False)) & (mq_eq_25['t_usomaq']==1)).astype(int)
mq_eq_25['t_maqalq'] = (mq_eq_25['P1208'].str.contains(pat=r'^Alquilado', na=False) & (mq_eq_25['t_usomaq']==1)).astype(int)
mq_eq_25=mq_eq_25.groupby('cod_prod', as_index=False)[['t_usomaq','t_usoequ','t_maqpro','t_maqalq']].max()

In [287]:
mq_eq=pd.concat([mq_eq_23,mq_eq_24,mq_eq_25], ignore_index=True)

In [289]:
mq_eq.to_stata('/content/drive/MyDrive/a_SENAMHI/OUT/mq_eq.dta')

In [296]:
# CREDITO

sf_23 = dta_dataframes[2023]['17_CAP900'].copy()
sf_23['cod_prod'] = sf_23['ANIO'].astype(str) + '_'+sf_23.NSEGM + sf_23.ID_PROD + sf_23.UA
sf_23['t_credito'] = (sf_23['P902'].str.contains('Sí', na=False) & ~sf_23['P903_9'].str.contains('Sí', na=False)).astype(int)
sf_23['t_seguro'] = sf_23['P905'].str.contains('Sí', na=False).astype(int)
sf_23['t_ahorro'] = sf_23['P907'].str.contains('Sí', na=False).astype(int)
sf_23 = sf_23.groupby('cod_prod', as_index=False)[['t_credito','t_seguro','t_ahorro']].max()

sf_24 = dta_dataframes[2024]['17_CAP900'].copy()
sf_24['cod_prod'] = sf_24['ANIO'].astype(str) + '_'+sf_24.NSEGM + sf_24.ID_PROD + sf_24.UA
sf_24['t_credito'] = (sf_24['P902'].str.contains('Sí', na=False) & ~sf_24['P903_9'].str.contains('Sí', na=False)).astype(int)
sf_24['t_seguro'] = sf_24['P905'].str.contains('Sí', na=False).astype(int)
sf_24['t_ahorro'] = sf_24['P907'].str.contains('Sí', na=False).astype(int)
sf_24 = sf_24.groupby('cod_prod', as_index=False)[['t_credito','t_seguro','t_ahorro']].max()

sf_25 = dta_dataframes[2025]['17_CAP900'].copy()
sf_25['cod_prod'] = sf_25['ANIO'].astype(str) + '_'+sf_25.NSEGM + sf_25.ID_PROD + sf_25.UA
sf_25['t_credito'] = (sf_25['P902'].str.contains('Sí', na=False) & ~sf_25['P903_9'].str.contains('Sí', na=False)).astype(int)
sf_25['t_seguro'] = sf_25['P905'].str.contains('Sí', na=False).astype(int)
sf_25['t_ahorro'] = sf_25['P907'].str.contains('Sí', na=False).astype(int)
sf_25 = sf_25.groupby('cod_prod', as_index=False)[['t_credito','t_seguro','t_ahorro']].max()

In [300]:
sf = pd.concat([sf_23, sf_24, sf_25], ignore_index=True)

In [302]:
sf.to_stata('/content/drive/MyDrive/a_SENAMHI/OUT/sf.dta')

In [327]:
vbpagri_23 = dta_dataframes[2023]['03_CAP200AB'].copy()
vbpagri_23['cod_prod'] = vbpagri_23['ANIO'].astype(str) + '_'+vbpagri_23.NSEGM + vbpagri_23.ID_PROD + vbpagri_23.UA
vbpagri_23['t_produc'] = pd.to_numeric(vbpagri_23['P219_CANT_1'],
        errors='coerce').fillna(0) + pd.to_numeric(vbpagri_23['P219_CANT_2'],
        errors='coerce').fillna(0) / 1000
vbpagri_23['t_prodkg'] = vbpagri_23['t_produc']*vbpagri_23['P219_EQUIV_KG'].fillna(0)
vbpagri_23['t_prventa'] = pd.to_numeric(vbpagri_23['P220_1_PREC_1'],
              errors='coerce').fillna(0) + pd.to_numeric(vbpagri_23['P220_1_PREC_2'],
              errors='coerce').fillna(0) / 1000
vbpagri_23['t_valventa'] = vbpagri_23['P220_1_VAL'].fillna(0)
# Calculate t_primplic, handling potential division by zero
vbpagri_23['t_primplic'] = vbpagri_23['t_valventa'] / vbpagri_23['t_prodkg']
vbpagri_23['t_primplic'] = vbpagri_23['t_primplic'].replace([np.inf, -np.inf], np.nan) # Replace inf with NaN
# Impute missing t_primplic with the mean of the column
mean_t_primplic = vbpagri_23['t_primplic'].mean()
vbpagri_23['t_primplic'] = vbpagri_23['t_primplic'].fillna(mean_t_primplic)
vbpagri_23['t_vbpagri'] = (vbpagri_23['t_prodkg'] * vbpagri_23['t_primplic']) / 1000
vbpagri_23 = vbpagri_23.groupby('cod_prod', as_index=False)['t_vbpagri'].sum()

vbpagri_24 = dta_dataframes[2024]['03_CAP200AB'].copy()
vbpagri_24['cod_prod'] = vbpagri_24['ANIO'].astype(str) + '_'+vbpagri_24.NSEGM + vbpagri_24.ID_PROD + vbpagri_24.UA
vbpagri_24['t_produc'] = pd.to_numeric(vbpagri_24['P219_CANT_1'],
        errors='coerce').fillna(0) + pd.to_numeric(vbpagri_24['P219_CANT_2'],
        errors='coerce').fillna(0) / 1000
vbpagri_24['t_prodkg'] = vbpagri_24['t_produc']*vbpagri_24['P219_EQUIV_KG'].fillna(0)
vbpagri_24['t_prventa'] = pd.to_numeric(vbpagri_24['P220_1_PREC_1'],
              errors='coerce').fillna(0) + pd.to_numeric(vbpagri_24['P220_1_PREC_2'],
              errors='coerce').fillna(0) / 1000
vbpagri_24['t_valventa'] = vbpagri_24['P220_1_VAL'].fillna(0)
# Calculate t_primplic, handling potential division by zero
vbpagri_24['t_primplic'] = vbpagri_24['t_valventa'] / vbpagri_24['t_prodkg']
vbpagri_24['t_primplic'] = vbpagri_24['t_primplic'].replace([np.inf, -np.inf], np.nan) # Replace inf with NaN
# Impute missing t_primplic with the mean of the column
mean_t_primplic = vbpagri_24['t_primplic'].mean()
vbpagri_24['t_primplic'] = vbpagri_24['t_primplic'].fillna(mean_t_primplic)
vbpagri_24['t_vbpagri'] = (vbpagri_24['t_prodkg'] * vbpagri_24['t_primplic']) / 1000
vbpagri_24 = vbpagri_24.groupby('cod_prod', as_index=False)['t_vbpagri'].sum()

vbpagri_25 = dta_dataframes[2025]['03_CAP200AB'].copy()
vbpagri_25['cod_prod'] = vbpagri_25['ANIO'].astype(str) + '_'+vbpagri_25.NSEGM + vbpagri_25.ID_PROD + vbpagri_25.UA
vbpagri_25['t_produc'] = pd.to_numeric(vbpagri_25['P219_CANT_1'],
        errors='coerce').fillna(0) + pd.to_numeric(vbpagri_25['P219_CANT_2'],
        errors='coerce').fillna(0) / 1000
vbpagri_25['t_prodkg'] = vbpagri_25['t_produc']*vbpagri_25['P219_EQUIV_KG'].fillna(0)
vbpagri_25['t_prventa'] = pd.to_numeric(vbpagri_25['P220_1_PREC_1'],
              errors='coerce').fillna(0) + pd.to_numeric(vbpagri_25['P220_1_PREC_2'],
              errors='coerce').fillna(0) / 1000
vbpagri_25['t_valventa'] = vbpagri_25['P220_1_VAL'].fillna(0)
# Calculate t_primplic, handling potential division by zero
vbpagri_25['t_primplic'] = vbpagri_25['t_valventa'] / vbpagri_25['t_prodkg']
vbpagri_25['t_primplic'] = vbpagri_25['t_primplic'].replace([np.inf, -np.inf], np.nan) # Replace inf with NaN
# Impute missing t_primplic with the mean of the column
mean_t_primplic = vbpagri_25['t_primplic'].mean()
vbpagri_25['t_primplic'] = vbpagri_25['t_primplic'].fillna(mean_t_primplic)
vbpagri_25['t_vbpagri'] = (vbpagri_25['t_prodkg'] * vbpagri_25['t_primplic']) / 1000
vbpagri_25 = vbpagri_25.groupby('cod_prod', as_index=False)['t_vbpagri'].sum()

In [333]:
vbpagri=pd.concat([vbpagri_23,vbpagri_24,vbpagri_25],ignore_index=True)

In [335]:
vbpagri.to_stata('/content/drive/MyDrive/a_SENAMHI/OUT/vbpagri.dta')